### **Project name**-Amazon_Prime_Video_movie and tv shows model

### **Project Summary**



The Amazon Prime Video Data Analysis project focuses on exploring movies and TV shows available on Amazon Prime Video. The project uses Python and data analysis techniques to understand the content library, identify trends, and discover insights about movies, TV shows, genres, release years, and ratings.
This project performs exploratory data analysis (EDA), trend identification, and predictive modeling on Amazon Prime Video's content library. By merging metadata from titles.csv (show/movie metadata) and credits.csv (cast and crew details), the project aims to evaluate content strategy, audience reception (IMDb/TMDB scores), and talent engagement on the platform.

My task is analyzes catalog balance, content runtime distributions, genre trends, rating patterns, and quality metrics over time.And Evaluates top directors/actors, casting overlap, and high-performing talent associations.

### **Problem Statement**
  

My task is to make a model that cantain rating and review.

### **GITHUB LINK**

In [1]:
https://github.com/deeksha1997nayak-hue/Deeksha_nayak.git

SyntaxError: invalid syntax (1716687807.py, line 1)

### **Let's Begain**

In [ ]:
!gdown --id 'https://drive.google.com/file/d/1pJEJ1dESmC4MbSB1_upFAs6qF3X-0La1/view?usp=sharing','https://drive.google.com/file/d/1tPXVEYBsb-RUlr15OVMaMdfWz2QVoPdt/view?usp=sharing'
pd.read_csv('/content/amazon_prime_titles.csv.zip')

### **1. Import libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")


### **2. Load datasets**

In [ ]:
titles = pd.read_csv("/content/amazon_prime_titles.csv.zip")
credits = pd.read_csv("/content/credits.csv.zip")

print("Titles shape:", titles.shape)
print("Credits shape:", credits.shape)

display(titles.head())
display(credits.head())


### **3. Inspect and clean data**

In [ ]:
print("Titles columns:", titles.columns.tolist())
print("Credits columns:", credits.columns.tolist())

print("\nTitles data types and non-null counts:")
titles.info()

print("\nMissing values in titles:")
display(titles.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nMissing values in credits:")
display(credits.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

# The titles dataset uses 'show_id' as the unique identifier
print("Duplicate title IDs before cleaning:", titles["show_id"].duplicated().sum())
print("Duplicate credit rows before cleaning:", credits.duplicated().sum())

# Keep one record per title ID and remove exact duplicate credit rows.
titles = titles.drop_duplicates(subset=["show_id"]).copy()
credits = credits.drop_duplicates().copy()

# Convert numeric columns that actually exist
numeric_columns = ["release_year"]
for col in numeric_columns:
    if col in titles.columns:
        titles[col] = pd.to_numeric(titles[col], errors="coerce")

for col in ["title", "type", "rating", "duration", "listed_in", "description"]:
    if col in titles.columns:
        titles[col] = titles[col].astype("string").str.strip()

titles = titles.dropna(subset=["show_id", "title"]).copy()

print("Cleaned titles shape:", titles.shape)
print("Cleaned credits shape:", credits.shape)

### **4. Exploratory data analysis and KPIs**

In [ ]:
print("Content types:")
display(titles["type"].value_counts(dropna=False).to_frame("title_count"))

print("Release year summary:")
display(titles["release_year"].describe().to_frame("release_year"))

# Extract numeric runtime from 'duration' for movies (e.g., '113 min' -> 113)
titles["runtime_minutes"] = pd.to_numeric(
    titles["duration"].astype(str).str.extract(r'(\d+)\s*min')[0],
    errors="coerce"
)

# Extract seasons count for TV shows (e.g., '4 Seasons' -> 4)
titles["seasons_count"] = pd.to_numeric(
    titles["duration"].astype(str).str.extract(r'(\d+)\s*Season')[0],
    errors="coerce"
)

total_titles = titles["show_id"].nunique()
type_counts = titles["type"].astype("string").str.upper().value_counts()
total_movies = int(type_counts.get("MOVIE", 0))
total_shows = int(type_counts.get("TV SHOW", 0))
avg_movie_runtime = titles.loc[
    titles["type"].astype("string").str.upper().eq("MOVIE"), "runtime_minutes"
].mean()

actor_rows = credits[credits["role"].astype("string").str.upper().eq("ACTOR")]
director_rows = credits[credits["role"].astype("string").str.upper().eq("DIRECTOR")]

kpis = pd.DataFrame({
    "Metric": [
        "Unique Titles",
        "Movies",
        "TV Shows",
        "Average Movie Runtime (minutes)",
        "Unique Actors in Credits",
        "Unique Directors in Credits"
    ],
    "Value": [
        total_titles,
        total_movies,
        total_shows,
        round(avg_movie_runtime, 2) if pd.notna(avg_movie_runtime) else np.nan,
        actor_rows["person_id"].nunique() if "person_id" in actor_rows.columns else np.nan,
        director_rows["person_id"].nunique() if "person_id" in director_rows.columns else np.nan
    ]
})
display(kpis)

### **5. Visualization 1 — Movies vs TV Shows**

In [2]:
plt.figure(figsize=(8, 5))
type_order = titles["type"].value_counts().index
sns.countplot(data=titles, x="type", order=type_order)
plt.title("Amazon Prime Video: Movies vs TV Shows")
plt.xlabel("Content type")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

### **6. Visualization 2 — Titles by release year**

In [ ]:
yearly_content = titles.dropna(subset=["release_year"]).groupby("release_year").size().sort_index()
plt.figure(figsize=(12, 5))
plt.plot(yearly_content.index, yearly_content.values, marker="o", linewidth=1)
plt.title("Titles by Release Year")
plt.xlabel("Release year")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.show()

print("Note: release year is not the same as the year a title was added to Prime Video.")


### **7. Visualization 3 — IMDb rating distribution**

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=titles, x="imdb_score", bins=20, kde=True)
plt.title("Distribution of IMDb Ratings")
plt.xlabel("IMDb score")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.show()


### **8. Visualization 4 — Movie runtime distribution**

In [ ]:
movie_mask = titles["type"].astype("string").str.upper().eq("MOVIE")
runtime_data = titles.loc[movie_mask].dropna(subset=["runtime_minutes"])
runtime_data = runtime_data[runtime_data["runtime_minutes"].between(1, 600)]

plt.figure(figsize=(10, 5))
sns.histplot(data=runtime_data, x="runtime_minutes", bins=30, kde=True)
plt.title("Distribution of Movie Runtime")
plt.xlabel("Runtime (minutes)")
plt.ylabel("Number of movies")
plt.tight_layout()
plt.show()

### **9. Visualization 5 — Top 10 genre**

In [ ]:
genre_data = titles.dropna(subset=["listed_in"])[["show_id", "listed_in"]].copy()
genre_data["genre"] = genre_data["listed_in"].astype(str).str.split(",")
genre_data = genre_data.explode("genre")
genre_data["genre"] = genre_data["genre"].astype(str).str.strip()
genre_data = genre_data[genre_data["genre"].ne("") & genre_data["genre"].ne("nan")]
genre_counts = genre_data["genre"].value_counts().head(10)

plt.figure(figsize=(11, 6))
sns.barplot(x=genre_counts.values, y=genre_counts.index)
plt.title("Top 10 Genres")
plt.xlabel("Title-genre associations")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

display(genre_counts.to_frame("title_genre_associations"))

### **10. Visualization 6 — Top 10 production countries**

In [ ]:
country_data = titles.dropna(subset=["country"])[["show_id", "country"]].copy()
country_data["country_clean"] = country_data["country"].astype(str).str.split(",")
country_data = country_data.explode("country_clean")
country_data["country_clean"] = country_data["country_clean"].astype(str).str.strip()
country_data = country_data[country_data["country_clean"].ne("") & country_data["country_clean"].ne("nan")]
country_counts = country_data["country_clean"].value_counts().head(10)

plt.figure(figsize=(11, 6))
sns.barplot(x=country_counts.values, y=country_counts.index)
plt.title("Top 10 Production Countries")
plt.xlabel("Title-country associations")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

display(country_counts.to_frame("title_country_associations"))

### **11. Visualization 7 — IMDb ratings by content type**

In [ ]:
# Since this dataset does not have IMDb ratings, we will plot the distribution of seasons for TV Shows instead
show_mask = titles["type"].astype("string").str.upper().eq("TV SHOW")
seasons_data = titles.loc[show_mask].dropna(subset=["seasons_count"])

if not seasons_data.empty:
    plt.figure(figsize=(9, 5))
    sns.countplot(data=seasons_data, x="seasons_count")
    plt.title("Distribution of TV Show Seasons")
    plt.xlabel("Number of Seasons")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No TV Show season information available to display.")

### **12. Visualization 8 — IMDb rating vs TMDB popularity**

In [ ]:
# Since this dataset does not have IMDb/TMDB scores, we will plot release year trends by content type
plt.figure(figsize=(10, 6))
sns.histplot(data=titles, x="release_year", hue="type", multiple="stack", bins=30)
plt.title("Release Year Distribution by Content Type")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.show()

### **13. Cast and director analysis**

In [ ]:
if {"role", "name"}.issubset(credits.columns):
    actor_data = credits[credits["role"].astype("string").str.upper().eq("ACTOR")]
    director_data = credits[credits["role"].astype("string").str.upper().eq("DIRECTOR")]

    top_actors = actor_data["name"].dropna().value_counts().head(10)
    top_directors = director_data["name"].dropna().value_counts().head(10)

    plt.figure(figsize=(11, 6))
    sns.barplot(x=top_actors.values, y=top_actors.index)
    plt.title("Top 10 Actors by Recorded Title Credits")
    plt.xlabel("Number of credits")
    plt.ylabel("Actor")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11, 6))
    sns.barplot(x=top_directors.values, y=top_directors.index)
    plt.title("Top 10 Directors by Recorded Title Credits")
    plt.xlabel("Number of credits")
    plt.ylabel("Director")
    plt.tight_layout()
    plt.show()

    print("Top actors:")
    display(top_actors.to_frame("credits"))
    print("Top directors:")
    display(top_directors.to_frame("credits"))
else:
    print("Expected columns 'role' and 'name' were not found in credits.")


### **14. Highest-rated titles and most popular titles**

In [ ]:
# Because this dataset lacks IMDb ratings, we'll list the top 10 oldest and top 10 newest titles
print("Top 10 oldest titles in the catalog:")
display(titles[["title", "type", "release_year", "rating"]].sort_values("release_year", ascending=True).head(10))

print("\nTop 10 newest titles in the catalog:")
display(titles[["title", "type", "release_year", "rating"]].sort_values("release_year", ascending=False).head(10))

### **15. Join titles and credits**

In [ ]:
# Since the credits.csv ID values (like 'ts20945') do not match titles.csv 'show_id' values (like 's1'),
# let's look at merging by matching actors/directors listed in titles['cast'] or titles['director'] columns
# or check if we can join on title strings by cleaning and aligning them.

# We will normalize title strings in both datasets to perform a matching merge.
# To do this, we can first map credits' title name. If credits does not have a title name column, we can build a link
# using the cast/director names listed in both datasets. Let's inspect the unique names to perform a robust match.

# First, let's create a normalized title lookup if possible, or link by director/cast string search.
print("Titles columns:", titles.columns.tolist())
print("Credits columns:", credits.columns.tolist())

# Let's perform a merge on 'title' if 'title' or another common identifier exists.
# Since 'credits' contains 'name' (of actors/directors), we can check if those names appear in titles 'cast' or 'director' columns.
# Let's write an optimized overlap check or a direct mapping if possible.
print("\nSample credits:")
display(credits.head(5))
print("\nSample titles with cast:")
display(titles[["show_id", "title", "director", "cast"]].dropna().head(5))

# Let's create a clean merge mapping names
merged = credits.copy()
# For demonstration and analysis, let's join where credit['name'] is in titles['cast'] or titles['director']
# To make it efficient, we explode the cast and director columns in titles and then merge on person name.
titles_exploded = titles.copy()
titles_exploded["cast_member"] = titles_exploded["cast"].astype(str).str.split(",")
titles_exploded = titles_exploded.explode("cast_member")
titles_exploded["cast_member"] = titles_exploded["cast_member"].astype(str).str.strip()

merged = credits.merge(titles_exploded, left_on="name", right_on="cast_member", how="inner")
print("\nSuccessfully merged on cast member name!")
print("Merged dataset shape:", merged.shape)
display(merged.head())

### **16. Automatically generated observations**

In [3]:
print("PROJECT FINDINGS")
print("=" * 50)

if not type_counts.empty:
    print(f"Most common content type: {type_counts.index[0]} ({type_counts.iloc[0]:,} titles)")

if 'genre_counts' in locals() and not genre_counts.empty:
    print(f"Most represented genre: {genre_counts.index[0]} ({genre_counts.iloc[0]:,} associations)")

if 'country_counts' in locals() and not country_counts.empty:
    print(f"Most represented country: {country_counts.index[0]} ({country_counts.iloc[0]:,} associations)")

if 'avg_movie_runtime' in locals() and pd.notna(avg_movie_runtime):
    print(f"Average Movie runtime: {avg_movie_runtime:.2f} minutes")

print("\nInterpret results as patterns in this dataset. The data does not directly measure viewing hours, subscriptions, revenue, or customer retention.")

PROJECT FINDINGS


NameError: name 'type_counts' is not defined

### **Conclusion**


The Amazon Prime Video Movies and TV Shows Data Analysis project demonstrates how Python can be used to explore and interpret entertainment data. By cleaning the datasets, performing exploratory data analysis, and creating visualizations, the project provides insights into the platform's content library and its development over time.